# 05b — CellRank: Pseudotime Root Sensitivity (audit re-run)

**Stage 4 audit · how much of Stage 4 depends on the choice of root cell?**

This is an **audit re-analysis, not a replacement.** `05_cellrank_fate_trajectory.ipynb`
stays as the original record; this notebook reruns the identical pipeline (same diffmap,
same kernel and GPCCA settings) with an alternate root and compares the results.

**Inputs**
- `data/Cherief_scRNA-seq/GSE244921_cluster8_sub.h5ad` — from `02`
- `data/pyscenic/cherief_regulons.csv`, `shared_diff_regulons.csv` — from `04`
- `data/cellrank/pyscenic_cellrank_tfap_overlap.csv` — from `05`, the list being stress-tested

**Outputs** — none written to disk; the findings are the printed comparisons and §5.

**Runs after:** `05` · **Feeds:** the Stage 4 caveats carried into `08`

**Environment:** analysis env (`environments/analysis.txt`). Run with the working directory set to `scripts/ipynb/` — every path below is relative to it.

---

**Why this notebook exists:** `05_cellrank_fate_trajectory.ipynb` already documents a caveat: DPT pseudotime (rooted at a Stromal cell, chosen because Stromal carries the imprinted-gene signature expected of an undifferentiated state) ranks Tenogenic-progenitor as *more* root-like (mean DPT 0.18) than Stromal itself (0.29) — which the notebook calls biologically implausible, and treats as a caveat confined to "trajectory ordering," while still trusting the resulting **fate probabilities** (82.6%→71.0% TSPC, 17.2%→28.9% T-FAP, innervated→denervated) as a valid "soft classification."

An independent audit questioned that separation: the fate probabilities come from GPCCA absorption probabilities computed on the *same* pseudotime-kernel transition matrix that produces the questionable ordering. If the root choice biases the flow direction, it plausibly biases the fate probabilities too, not just the ordering label.

**Method:** rerun the identical pipeline with an alternate root — a Tenogenic-progenitor cell with minimum DC1, i.e. the cell the diffusion map's own ranking would have picked if not overridden by the biological prior that Stromal should be the root — and compare the resulting fate probabilities, and then the 119-gene candidate list, to the original.


## 0. Setup

Imports, paths and output directories.


In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr
from cellrank.kernels import PseudotimeKernel
from cellrank.estimators import GPCCA
from pathlib import Path

sc.settings.verbosity = 1

ROOT    = Path('../..')
DATA_SC = ROOT / 'data' / 'Cherief_scRNA-seq'

adata = sc.read(DATA_SC / 'GSE244921_cluster8_sub.h5ad')
print(adata.obs['cell_type'].value_counts())

sc.tl.diffmap(adata, n_comps=15)

cell_type
Stromal                 1388
T-FAP                   1245
Tenogenic-progenitor     816
TSPC                     451
Name: count, dtype: int64


## 1. Two root choices

- **Original:** Stromal cell with minimum DC1 (biological prior: Stromal carries the imprinted-gene undifferentiated signature)
- **Alternate:** Tenogenic-progenitor cell with minimum DC1 (what the diffusion map itself ranks as most root-like, per the caveat already on record)

In [2]:
stromal_mask = adata.obs['cell_type'] == 'Stromal'
dc1 = adata.obsm['X_diffmap'][stromal_mask, 1]
root_cell_orig = adata.obs.index[stromal_mask][np.argmin(dc1)]
print('Original root (Stromal, min DC1):', root_cell_orig)

teno_mask = adata.obs['cell_type'] == 'Tenogenic-progenitor'
dc1_teno = adata.obsm['X_diffmap'][teno_mask, 1]
root_cell_alt = adata.obs.index[teno_mask][np.argmin(dc1_teno)]
print('Alternate root (Tenogenic-progenitor, min DC1):', root_cell_alt)

Original root (Stromal, min DC1): GGGTTTAAGCCTAACT-1
Alternate root (Tenogenic-progenitor, min DC1): GCAGTTACATGCGGTC-1


## 2. Rerun the identical DPT -> PseudotimeKernel -> GPCCA -> fate-probability pipeline for each root

`n_jobs=1, show_progress_bar=False` on `compute_transition_matrix` -- avoids a Windows-specific multiprocessing issue with the default `loky` backend spawning a `Manager()` process when this is run as a plain script outside a `__main__` guard; not needed for correctness, only for this environment.

In [3]:
def run_cellrank(adata, root_cell, label):
    a = adata.copy()
    a.uns['iroot'] = a.obs.index.get_loc(root_cell)
    sc.tl.dpt(a, n_dcs=10)
    print(f'\n[{label}] Mean DPT by cell type:')
    print(a.obs.groupby('cell_type', observed=True)['dpt_pseudotime'].mean().sort_values())

    pk = PseudotimeKernel(a, time_key='dpt_pseudotime')
    pk.compute_transition_matrix(threshold_scheme='soft', nu=0.5, n_jobs=1, show_progress_bar=False)

    g = GPCCA(pk)
    g.compute_schur(n_components=6)
    try:
        g.compute_macrostates(n_states=5, cluster_key='cell_type')
    except Exception as e:
        print(f'[{label}] n_states=5 failed ({e}), trying n_states=4')
        g.compute_macrostates(n_states=4, cluster_key='cell_type')

    print(f'[{label}] Macrostates:', g.macrostates.cat.categories.tolist())
    # NOTE: fixed vs. first pass -- the alternate root splits TSPC into TSPC_1/TSPC_2
    # (mirroring how T-FAP already splits into T-FAP_1/T-FAP_2 under the original root).
    # The first version of this check only matched the exact string 'TSPC' and silently
    # dropped TSPC_1/TSPC_2 from the terminal set, which made fate probabilities fail to
    # sum to 1 (most mass was absorbing into the excluded TSPC states). Use startswith for both.
    terminal_states = [s for s in g.macrostates.cat.categories if s.startswith('TSPC') or s.startswith('T-FAP')]
    print(f'[{label}] Terminal states:', terminal_states)
    if not terminal_states:
        print(f'[{label}] WARNING: no TSPC/T-FAP terminal states recovered -- skipping fate probs')
        return None, None, g, a

    g.set_terminal_states(terminal_states)
    g.compute_fate_probabilities(solver='direct', check_sum_tol=0.05)

    lin = a.obsm['lineages_fwd']
    fate_df = pd.DataFrame(np.array(lin), columns=lin.names, index=a.obs.index)
    fate_df['condition'] = a.obs['condition'].values
    fate_df['cell_type'] = a.obs['cell_type'].values

    tfap_cols = [s for s in terminal_states if s.startswith('T-FAP')]
    tspc_cols = [s for s in terminal_states if s.startswith('TSPC')]
    fate_df['T-FAP_total'] = fate_df[tfap_cols].sum(axis=1) if len(tfap_cols) > 1 else fate_df[tfap_cols[0]]
    fate_df['TSPC_total'] = fate_df[tspc_cols].sum(axis=1) if len(tspc_cols) > 1 else fate_df[tspc_cols[0]]

    by_cond = fate_df.groupby('condition', observed=True)[['TSPC_total', 'T-FAP_total']].mean().round(3)
    print(f'[{label}] Fate probability by condition:')
    print(by_cond)
    return fate_df, by_cond, g, a


fate_orig, summary_orig, g_orig, a_orig = run_cellrank(adata, root_cell_orig, 'ORIGINAL (Stromal root)')


[ORIGINAL (Stromal root)] Mean DPT by cell type:
cell_type
Tenogenic-progenitor    0.179656
Stromal                 0.292036
T-FAP                   0.366106
TSPC                    0.658067
Name: dpt_pseudotime, dtype: float32
INFO     Computing transition matrix based on pseudotime                                                           


INFO         Finish (0.27s)                                                                                        


WARNING  Unable to import `petsc4py` or `slepc4py`. Using `method='brandts'`                                       


WARNING  For `method='brandts'`, dense matrix is required. Densifying                                              


INFO     Computing Schur decomposition                                                                             


...

INFO     Computing fate probabilities                                                                              


WARNING  Unable to import petsc4py. For installation, please refer to:                                             
         https://petsc4py.readthedocs.io/en/stable/install.html.                                                   
         Defaulting to `'gmres'` solver.                                                                           


  0%|          | 0/3 [00:00<?, ?/s]

INFO     Adding `adata.obsm['lineages_fwd']`                                                                       
                `.fate_probabilities`                                                                              
             Finish (0.87s)                                                                                        


[ORIGINAL (Stromal root)] Fate probability by condition:
           TSPC_total  T-FAP_total
condition                         
TrkAF592A       0.710        0.289
TrkAWT          0.826        0.172


In [4]:
fate_alt, summary_alt, g_alt, a_alt = run_cellrank(adata, root_cell_alt, 'ALTERNATE (Tenogenic-progenitor root)')


[ALTERNATE (Tenogenic-progenitor root)] Mean DPT by cell type:
cell_type
Tenogenic-progenitor    0.201622
Stromal                 0.360107
T-FAP                   0.432375
TSPC                    0.677039
Name: dpt_pseudotime, dtype: float32
INFO     Computing transition matrix based on pseudotime                                                           


INFO         Finish (0.30s)                                                                                        


WARNING  Unable to import `petsc4py` or `slepc4py`. Using `method='brandts'`                                       


WARNING  For `method='brandts'`, dense matrix is required. Densifying                                              


INFO     Computing Schur decomposition                                                                             


...

[ALTERNATE (Tenogenic-progenitor root)] Macrostates: ['Stromal', 'T-FAP_1', 'T-FAP_2', 'TSPC_1', 'TSPC_2']
[ALTERNATE (Tenogenic-progenitor root)] Terminal states: ['T-FAP_1', 'T-FAP_2', 'TSPC_1', 'TSPC_2']
INFO     Adding `adata.obs['term_states_fwd']`                                                                     
                `adata.obs['term_states_fwd_probs']`                                                               
                `.terminal_states`                                                                                 
                `.terminal_states_probabilities`                                                                   
                `.terminal_states_memberships                                                                      
             Finish`                                                                                               


INFO     Computing fate probabilities                                                                              


  0%|          | 0/4 [00:00<?, ?/s]

INFO     Adding `adata.obsm['lineages_fwd']`                                                                       
                `.fate_probabilities`                                                                              
             Finish (0.42s)                                                                                        


[ALTERNATE (Tenogenic-progenitor root)] Fate probability by condition:
           TSPC_total  T-FAP_total
condition                         
TrkAF592A       0.641        0.359
TrkAWT          0.767        0.232


## 3. Comparison

In [5]:
print('Original (Stromal root):')
print(summary_orig)
print()
print('Alternate (Tenogenic-progenitor root):')
print(summary_alt)

Original (Stromal root):
           TSPC_total  T-FAP_total
condition                         
TrkAF592A       0.710        0.289
TrkAWT          0.826        0.172

Alternate (Tenogenic-progenitor root):
           TSPC_total  T-FAP_total
condition                         
TrkAF592A       0.641        0.359
TrkAWT          0.767        0.232


## 4. The check that actually matters for Stage 7: does the 119-gene candidate list change?

Sections 2-3 tested whether the *fate probability percentages* are root-sensitive. That was the wrong sensitivity test for Stage 7 — Stage 7 doesn't consume fate probabilities at all. It consumes the **119-gene candidate feature list** in `data/cellrank/pyscenic_cellrank_tfap_overlap.csv`, which comes from intersecting CellRank's **lineage driver genes** (not fate probabilities) with pySCENIC T-FAP regulon targets. This section reruns that specific computation under the alternate root and checks how much the 119-gene list actually moves — replicating §5 (lineage drivers) and §6 (pySCENIC × CellRank overlap) of `05_cellrank_fate_trajectory.ipynb` exactly, once per root.

In [6]:
import ast
import re

DATA_PY = ROOT / 'data' / 'pyscenic'
DATA_CR = ROOT / 'data' / 'cellrank'

def top200_driver_union(g, adata_run, lineages, label):
    drivers = g.compute_lineage_drivers(lineages=lineages, use_raw=True, return_drivers=True)
    union = set()
    for lt in lineages:
        corr_col, qval_col = f'{lt}_corr', f'{lt}_qval'
        top = drivers[drivers[qval_col] < 0.05].sort_values(corr_col, ascending=False).head(200)
        union |= set(top.index)
    print(f'[{label}] top-200 driver union across {lineages}: {len(union)} genes')
    return union, drivers

tfap_states_orig = [s for s in g_orig.macrostates.cat.categories if s.startswith('T-FAP')]
tfap_states_alt = [s for s in g_alt.macrostates.cat.categories if s.startswith('T-FAP')]

orig_driver_union, drivers_orig = top200_driver_union(g_orig, a_orig, tfap_states_orig, 'ORIGINAL')
alt_driver_union, drivers_alt = top200_driver_union(g_alt, a_alt, tfap_states_alt, 'ALTERNATE')

overlap_drivers = orig_driver_union & alt_driver_union
print(f'\nDriver-gene overlap between roots: {len(overlap_drivers)} / {len(orig_driver_union)} original genes '
      f'({100*len(overlap_drivers)/len(orig_driver_union):.0f}%) also appear under the alternate root')

INFO     Adding `adata.raw.varm['terminal_lineage_drivers']`                                                       
                `.lineage_drivers`                                                                                 
             Finish (0.37s)                                                                                        


[ORIGINAL] top-200 driver union across ['T-FAP_1', 'T-FAP_2']: 374 genes


INFO     Adding `adata.raw.varm['terminal_lineage_drivers']`                                                       
                `.lineage_drivers`                                                                                 
             Finish (0.38s)                                                                                        


[ALTERNATE] top-200 driver union across ['T-FAP_1', 'T-FAP_2']: 369 genes

Driver-gene overlap between roots: 347 / 374 original genes (93%) also appear under the alternate root


In [7]:
# Reproduce the exact pySCENIC T-FAP target-gene set used in §6 of 05_cellrank_fate_trajectory.ipynb
# (Cherief-only regulon targets, v1 15-TF list)
regulons_raw = pd.read_csv(DATA_PY / 'cherief_regulons.csv', index_col=0, header=[0, 1])
shared = pd.read_csv(DATA_PY / 'shared_diff_regulons.csv')
tfap_regulons = shared[shared['cell_type'] == 'T-FAP']['regulon'].tolist()

target_col = ('Enrichment', 'TargetGenes')
clean_reg = regulons_raw[regulons_raw.index != 'TF']

all_tfap_targets = set()
for reg in tfap_regulons:
    tf_name = re.sub(r'\([+-]\)$', '', reg)
    rows = clean_reg[clean_reg.index == tf_name]
    for val in rows[target_col].dropna():
        try:
            pairs = ast.literal_eval(val) if isinstance(val, str) else val
            all_tfap_targets.update(g for g, _ in pairs)
        except Exception:
            pass
print(f'pySCENIC T-FAP target genes (same set used for the original 119-gene overlap): {len(all_tfap_targets)}')

# The actual candidate-gene-list computation, once per root
orig_119_equivalent = orig_driver_union & all_tfap_targets
alt_119_equivalent = alt_driver_union & all_tfap_targets

print(f'\nORIGINAL root -> candidate gene list: {len(orig_119_equivalent)} genes')
print(f'ALTERNATE root -> candidate gene list: {len(alt_119_equivalent)} genes')

# Sanity check against the file actually sitting in data/cellrank/ (should match "ORIGINAL" count)
existing_119 = set(pd.read_csv(DATA_CR / 'pyscenic_cellrank_tfap_overlap.csv')['gene'])
print(f'\nExisting data/cellrank/pyscenic_cellrank_tfap_overlap.csv: {len(existing_119)} genes')
print(f'Matches ORIGINAL recomputation: {existing_119 == orig_119_equivalent}')

candidate_overlap = orig_119_equivalent & alt_119_equivalent
candidate_union = orig_119_equivalent | alt_119_equivalent
jaccard = len(candidate_overlap) / len(candidate_union) if candidate_union else float('nan')
print(f'\nCandidate-list overlap between roots: {len(candidate_overlap)} genes shared '
      f'({100*len(candidate_overlap)/len(orig_119_equivalent):.0f}% of the original 119-gene list survives under the alternate root)')
print(f'Jaccard similarity: {jaccard:.2f}')
print(f'Genes in ORIGINAL list only (lost under alternate root): {len(orig_119_equivalent - alt_119_equivalent)}')
print(f'Genes in ALTERNATE list only (gained, not in original 119): {len(alt_119_equivalent - orig_119_equivalent)}')

pySCENIC T-FAP target genes (same set used for the original 119-gene overlap): 1682

ORIGINAL root -> candidate gene list: 119 genes
ALTERNATE root -> candidate gene list: 119 genes

Existing data/cellrank/pyscenic_cellrank_tfap_overlap.csv: 119 genes
Matches ORIGINAL recomputation: True

Candidate-list overlap between roots: 112 genes shared (94% of the original 119-gene list survives under the alternate root)
Jaccard similarity: 0.89
Genes in ORIGINAL list only (lost under alternate root): 7
Genes in ALTERNATE list only (gained, not in original 119): 7


## 5. Conclusion for Stage 4/6/7

**Fate probabilities (Sections 2-3, the wrong test for Stage 7 but still relevant to Stage 4/6 narrative):**

| | TSPC (innervated) | T-FAP (innervated) | TSPC (denervated) | T-FAP (denervated) | Shift (innerv.→denerv.) |
|---|---|---|---|---|---|
| Original (Stromal root) | 82.6% | 17.2% | 71.0% | 28.9% | TSPC −11.6pp / T-FAP +11.7pp |
| Alternate (Tenogenic-progenitor root) | 76.7% | 23.2% | 64.1% | 35.9% | TSPC −12.6pp / T-FAP +12.7pp |

Absolute percentages are *not* root-invariant (~5-7pp shift), but the direction and magnitude of the innervated→denervated change is robust either way.

**The 119-gene Stage 7 candidate list (Section 4, the test that actually matters):**

- Recomputing the exact candidate-list pipeline (top-200 T-FAP lineage drivers ∩ pySCENIC T-FAP targets) under the alternate root reproduces the original 119-gene file exactly when using the original root (sanity check passed).
- Under the alternate root: **112 of the 119 genes (94%) are unchanged**, Jaccard similarity 0.89. 7 genes drop out, 7 different genes come in.
- **This is the one that matters for Stage 7, and it's good news: the candidate gene list is robust to the pseudotime-root concern**, even though the fate-probability percentages and the trajectory ordering are not. Stage 7 can proceed on `data/cellrank/pyscenic_cellrank_tfap_overlap.csv` as-is without needing to rebuild it against an alternate root.

**Overall takeaway:** two different sensitivities, two different answers. Summary statistics computed *across the whole population* (fate probability means) shift noticeably with root choice. The specific *gene list* used for Stage 7 feature selection is much more stable (94% overlap) — likely because genes strongly correlated with T-FAP identity stay strongly correlated regardless of exactly where the trajectory is anchored, even when the population-level probability of ending up there shifts by ~10 points.